# Notebook 05: U.S. Lucas (1980) Appendix

Descriptive appendix. Two-sided exponential MA filter (β=0.9) applied to annual FRED data. M1 estimated 1960–2019 (truncated before May 2020 definitional break); M2 estimated 1960–2024.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

warnings.filterwarnings("ignore")


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "02_data/analysis_ready/macro_growth_merged.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find project root.")


ROOT  = find_project_root(Path.cwd().resolve())
RAW   = ROOT / "02_data/raw"
FIGS  = ROOT / "04_current_results/figures"
TABS  = ROOT / "04_current_results/tables"
FIGS.mkdir(parents=True, exist_ok=True)
TABS.mkdir(parents=True, exist_ok=True)

BETA       = 0.9   # Lucas (1980) main filter parameter
M1_END     = 2019  # last year before 2020 M1 definitional break
M2_END     = 2024
START_YEAR = 1960
HAC_LAGS   = 4     # Newey-West lags: compensates for serial correlation induced by the filter

In [2]:
# Download M1SL from FRED public endpoint if not already present (no API key required)
import requests

m1_path = RAW / "fred_M1SL.csv"
if not m1_path.exists():
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=M1SL"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    m1_path.write_bytes(resp.content)
    print(f"Downloaded M1SL -> {m1_path}")
else:
    print(f"Found {m1_path}")

Found /Users/stevenchung/Desktop/P12B_File/Monetary_Panel/02_data/raw/fred_M1SL.csv


In [3]:
def load_annual(file_name: str, value_col: str) -> pd.Series:
    df = pd.read_csv(RAW / file_name, parse_dates=["observation_date"])
    return df.groupby(df["observation_date"].dt.year)[value_col].mean()


m1    = load_annual("fred_M1SL.csv",     "M1SL")
m2    = load_annual("fred_M2SL.csv",     "M2SL")
cpi   = load_annual("fred_CPIAUCSL.csv", "CPIAUCSL")
tbill = load_annual("fred_TB3MS.csv",    "TB3MS")

m1_growth = np.log(m1  / m1.shift(1)).rename("m1_growth")
m2_growth = np.log(m2  / m2.shift(1)).rename("m2_growth")
inflation = np.log(cpi / cpi.shift(1)).rename("inflation")
tbill_dec = (tbill / 100).rename("tbill_dec")

# M1 dataset: 1960-2019 (pre-break)
us_m1 = pd.DataFrame({"money": m1_growth, "inflation": inflation, "tbill": tbill_dec}).dropna()
us_m1 = us_m1.loc[START_YEAR:M1_END]

# M2 dataset: 1960-2024
us_m2 = pd.DataFrame({"money": m2_growth, "inflation": inflation, "tbill": tbill_dec}).dropna()
us_m2 = us_m2.loc[START_YEAR:M2_END]

print(f"M1 raw obs: {len(us_m1)}  ({us_m1.index.min()}-{us_m1.index.max()})")
print(f"M2 raw obs: {len(us_m2)}  ({us_m2.index.min()}-{us_m2.index.max()})")

M1 raw obs: 60  (1960-2019)
M2 raw obs: 65  (1960-2024)


In [ ]:
def lucas_filter(series: pd.Series, beta: float = BETA) -> pd.Series:
    """Two-sided exponential MA per Lucas (1980) eq. (1), boundary-normalised, first/last 2 obs NaN."""
    alpha = (1 - beta) / (1 + beta)
    x = series.values.astype(float)
    n = len(x)

    f = np.zeros(n); f[0] = x[0]
    for t in range(1, n):
        f[t] = beta * f[t - 1] + x[t]

    b = np.zeros(n); b[-1] = x[-1]
    for t in range(n - 2, -1, -1):
        b[t] = beta * b[t + 1] + x[t]

    fw = np.zeros(n); fw[0] = 1.0
    for t in range(1, n):
        fw[t] = beta * fw[t - 1] + 1.0

    bw = np.zeros(n); bw[-1] = 1.0
    for t in range(n - 2, -1, -1):
        bw[t] = beta * bw[t + 1] + 1.0

    filtered = alpha * (f + b - x) / (alpha * (fw + bw - 1.0))

    result = pd.Series(filtered, index=series.index, dtype=float)
    result.iloc[:2]  = np.nan
    result.iloc[-2:] = np.nan
    return result


def apply_lucas_filter(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in ["money", "inflation", "tbill"]:
        out[f"{col}_ma"] = lucas_filter(df[col])
    return out.dropna()


m1_f = apply_lucas_filter(us_m1)
m2_f = apply_lucas_filter(us_m2)

print(f"M1 filtered obs: {len(m1_f)}  ({m1_f.index.min()}-{m1_f.index.max()})")
print(f"M2 filtered obs: {len(m2_f)}  ({m2_f.index.min()}-{m2_f.index.max()})")

## M1 Estimation: 1960–2019

Lucas's original aggregate. Expected under the quantity theory: scatter falls on a 45° line.

In [5]:
def ols_hac(df, y_col, x_col):
    fit = smf.ols(f"{y_col} ~ {x_col}", data=df).fit()
    x   = sm.add_constant(df[[x_col]])
    hac = sm.OLS(df[y_col], x).fit(cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS})
    return {
        "slope": round(float(fit.params[x_col]),  4),
        "r2":    round(float(fit.rsquared),        4),
        "p_hac": round(float(hac.pvalues[x_col]),  4),
        "n":     len(df),
        "fit":   fit,
    }


m1_infl = ols_hac(m1_f, "inflation_ma", "money_ma")
m1_tb   = ols_hac(m1_f, "tbill_ma",     "money_ma")

print("M1 results (Lucas filter beta=0.9):")
print(f"  M1 -> Inflation:  slope={m1_infl['slope']:.3f}  R2={m1_infl['r2']:.3f}  p_HAC={m1_infl['p_hac']:.4f}  n={m1_infl['n']}")
print(f"  M1 -> T-bill:     slope={m1_tb['slope']:.3f}  R2={m1_tb['r2']:.3f}  p_HAC={m1_tb['p_hac']:.4f}  n={m1_tb['n']}")
print()
print("Slope near zero: sweep-account distortions (1990s-2000s) and QE-era M1 surge (2009-2019)")
print("with low inflation pull the filtered series apart. See Interpretation cell.")

M1 results (Lucas filter beta=0.9):
  M1 -> Inflation:  slope=0.079  R2=0.003  p_HAC=0.8570  n=56
  M1 -> T-bill:     slope=-0.273  R2=0.015  p_HAC=0.6827  n=56

Slope near zero: sweep-account distortions (1990s-2000s) and QE-era M1 surge (2009-2019)
with low inflation pull the filtered series apart. See Interpretation cell.


In [ ]:
def lucas_scatter(ax, df, x_col, y_col, result, xlabel, ylabel, title):
    x_pct = df[x_col] * 100
    y_pct = df[y_col] * 100
    sc = ax.scatter(x_pct, y_pct, c=df.index, cmap="RdYlBu_r", alpha=0.85, s=42, zorder=3)
    x_range = np.linspace(float(x_pct.min()), float(x_pct.max()), 100)
    ic = result["fit"].params["Intercept"] * 100
    sl = result["slope"]
    ax.plot(x_range, ic + sl * x_range, color="firebrick", lw=2,
            label=f"OLS  slope={sl:.2f}  R²={result['r2']:.2f}")
    mn = min(float(x_pct.min()), float(y_pct.min()))
    mx = max(float(x_pct.max()), float(y_pct.max()))
    ax.plot([mn, mx], [mn, mx], color="grey", lw=1.2, ls="--", label="45° (slope=1, Lucas prediction)")
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    return sc


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc = lucas_scatter(axes[0], m1_f, "money_ma", "inflation_ma", m1_infl,
                   "M1 growth, Lucas filter β=0.9 (%)",
                   "CPI inflation, Lucas filter β=0.9 (%)",
                   "Illustration I: M1 vs Inflation\n1962–2017")
lucas_scatter(axes[1], m1_f, "money_ma", "tbill_ma", m1_tb,
              "M1 growth, Lucas filter β=0.9 (%)",
              "3-month T-bill, Lucas filter β=0.9 (%)",
              "Illustration II: M1 vs T-bill (Fisher)\n1962–2017")
plt.colorbar(sc, ax=axes[0], label="Year")
plt.suptitle("M1 relationship breaks down over 1962–2017 — see Interpretation", fontsize=9, color="#888")
plt.tight_layout()
fig.savefig(FIGS / "lucas_m1_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved -> lucas_m1_scatter.png")

## M2 Estimation: 1960–2024

Same Lucas filter applied to M2 for the full modern sample.

In [7]:
m2_infl = ols_hac(m2_f, "inflation_ma", "money_ma")
m2_tb   = ols_hac(m2_f, "tbill_ma",     "money_ma")

print("M2 results (Lucas filter beta=0.9):")
print(f"  M2 -> Inflation:  slope={m2_infl['slope']:.3f}  R2={m2_infl['r2']:.3f}  p_HAC={m2_infl['p_hac']:.4f}  n={m2_infl['n']}")
print(f"  M2 -> T-bill:     slope={m2_tb['slope']:.3f}  R2={m2_tb['r2']:.3f}  p_HAC={m2_tb['p_hac']:.4f}  n={m2_tb['n']}")

# combined summary for report
summary = pd.DataFrame([
    {"aggregate":"M1", "sample":f"{m1_f.index.min()}-{m1_f.index.max()}",
     "relationship":"M1->Inflation", "slope":m1_infl["slope"], "R2":m1_infl["r2"],
     "p_HAC":m1_infl["p_hac"], "n":m1_infl["n"]},
    {"aggregate":"M1", "sample":f"{m1_f.index.min()}-{m1_f.index.max()}",
     "relationship":"M1->T-bill",    "slope":m1_tb["slope"],   "R2":m1_tb["r2"],
     "p_HAC":m1_tb["p_hac"],   "n":m1_tb["n"]},
    {"aggregate":"M2", "sample":f"{m2_f.index.min()}-{m2_f.index.max()}",
     "relationship":"M2->Inflation", "slope":m2_infl["slope"], "R2":m2_infl["r2"],
     "p_HAC":m2_infl["p_hac"], "n":m2_infl["n"]},
    {"aggregate":"M2", "sample":f"{m2_f.index.min()}-{m2_f.index.max()}",
     "relationship":"M2->T-bill",    "slope":m2_tb["slope"],   "R2":m2_tb["r2"],
     "p_HAC":m2_tb["p_hac"],   "n":m2_tb["n"]},
])
summary.to_csv(TABS / "lucas_us_summary.csv", index=False)
print()
print(summary.to_string(index=False))

M2 results (Lucas filter beta=0.9):
  M2 -> Inflation:  slope=1.041  R2=0.623  p_HAC=0.0000  n=61
  M2 -> T-bill:     slope=1.357  R2=0.377  p_HAC=0.0003  n=61

aggregate    sample  relationship   slope     R2  p_HAC  n
       M1 1962-2017 M1->Inflation  0.0786 0.0028 0.8570 56
       M1 1962-2017    M1->T-bill -0.2725 0.0152 0.6827 56
       M2 1962-2022 M2->Inflation  1.0410 0.6226 0.0000 61
       M2 1962-2022    M2->T-bill  1.3573 0.3771 0.0003 61


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc = lucas_scatter(axes[0], m2_f, "money_ma", "inflation_ma", m2_infl,
                   "M2 growth, Lucas filter β=0.9 (%)",
                   "CPI inflation, Lucas filter β=0.9 (%)",
                   "Illustration I: M2 vs Inflation\n1962–2022")
lucas_scatter(axes[1], m2_f, "money_ma", "tbill_ma", m2_tb,
              "M2 growth, Lucas filter β=0.9 (%)",
              "3-month T-bill, Lucas filter β=0.9 (%)",
              "Illustration II: M2 vs T-bill (Fisher)\n1962–2022")
plt.colorbar(sc, ax=axes[0], label="Year")
plt.tight_layout()
fig.savefig(FIGS / "lucas_m2_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved -> lucas_m2_scatter.png")

## Pre-QE vs Post-QE (M2)

Lucas-filtered M2 split at 2008. Pre-2008 slope is closer to 1; post-QE era is flatter.

In [ ]:
pre  = m2_f[m2_f.index <= 2007]
post = m2_f[m2_f.index >= 2008]

fig, ax = plt.subplots(figsize=(8, 6))

def add_era(df, color, marker, era_label):
    x = df["money_ma"] * 100
    y = df["inflation_ma"] * 100
    sl, ic, rv, _, _ = stats.linregress(x.values, y.values)
    ax.scatter(x, y, color=color, marker=marker, s=62, zorder=3,
               edgecolors="white", lw=0.5,
               label=f"{era_label}  (slope={sl:.2f}, R²={rv**2:.2f})")
    xfit = np.linspace(float(x.min()), float(x.max()), 100)
    ax.plot(xfit, ic + sl * xfit, color=color, lw=2.2, alpha=0.85)


add_era(pre,  "#2166ac", "o", "Pre-QE  1962–2007")
add_era(post, "#d6604d", "s", "Post-QE  2008–2022")

for year, row in m2_f[m2_f.index >= 2020].iterrows():
    ax.annotate(str(year), (row["money_ma"]*100, row["inflation_ma"]*100),
                fontsize=7.5, color="#b2182b",
                textcoords="offset points", xytext=(5, 3))

ax.set_xlabel("M2 growth, Lucas filter β=0.9 (%)", fontsize=11)
ax.set_ylabel("CPI inflation, Lucas filter β=0.9 (%)", fontsize=11)
ax.set_title("US: M2 vs Inflation Across Two Monetary Eras\n"
             "Lucas filter β=0.9 · 1962–2022 · COVID years labelled",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=9.5, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig.savefig(FIGS / "us_m2_inflation_two_eras.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved -> us_m2_inflation_two_eras.png")

## Interpretation

- **M1 (1962–2017)**: slope≈0.08, R²≈0.003 — breaks down (sweep-account distortions 1990s; QE-era M1 surge 2009–2019 with low inflation).
- **M2 (1962–2022)**: slope≈1.04, R²≈0.62 — close to Lucas's 1.0. Pre-2008 slope≈0.87; post-2008 slope≈0.39 (mirrors Notebook 02 cross-country finding).
- Descriptive appendix only.